# ConeVision: YOLOv8n Cone Detector Training

Fine-tunes a COCO-pretrained YOLOv8n on a traffic-cone dataset and exports it to ONNX for C++ inference. Runs on a Colab T4 GPU.

## Environment

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow

## Dataset

Traffic-cone dataset from Roboflow Universe, in YOLOv8 format. The API key is read from Colab Secrets (`ROBOFLOW_API_KEY`).

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

WORKSPACE = "project-k84yn"
PROJECT = "traffic-cone-ry08d"
VERSION = 3

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")

## Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="cones",
)

## Evaluation

In [ ]:
best = YOLO("runs/detect/cones/weights/best.pt")

for split in ["val", "test"]:
    m = best.val(split=split)
    print(f"[{split}] mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f} | "
          f"precision {m.box.mp:.3f} | recall {m.box.mr:.3f}")

In [ ]:
import glob
from IPython.display import Image, display

img = sorted(glob.glob(f"{dataset.location}/test/images/*"))[0]
best.predict(img, save=True, name="demo", exist_ok=True)
display(Image(filename=sorted(glob.glob("runs/detect/demo/*"))[0]))

## Export

In [ ]:
best.export(format="onnx", imgsz=640)

from google.colab import files
files.download("runs/detect/cones/weights/best.pt")
files.download("runs/detect/cones/weights/best.onnx")